# FLAG 2027 - Notebook 37 RUN-ALL: what the FAME winners did that we have not tried (research round 7)

A four-way literature sweep on 5 Nov (FAME 2024/2026 winners, 2022-2026 loss and head papers, attribute and language work, representation tricks; summary in `research/round7_2026-11-05.md`) found five things used by the top honest systems that we have never tested. All are training-side, per-trial at inference, no test statistics.

| Name | What | Precedent |
|---|---|---|
| `F_agv` | concatenate the frozen age-gender ViT embedding (CLS, 768-d) to CLIP on the face side | FAME 2026 winner (Simic et al.) used exactly this model next to VGGFace; age is CLIP's weakest attribute |
| `L_sf2` | SphereFace2-style binary loss per prototype instead of softmax CE on the fixed prototypes | Han et al. ICASSP 2023; targets closed-set training with open-set testing |
| `L_hinge` | margin-relaxed cosine hinge between the scored face and voice embeddings (no gradient inside 30 degrees) | cross-modal distillation 2025: the hard MSE form fails, the hinge form works |
| `H_pair` | small pair classifier on [f, v, f*v, abs(f-v)], BCE on in-batch pairs, scored by its logit | FAME 2024 winner (HLT) scored this way |
| `X_mix` | within-identity latent mixup with a shared lambda on face and voice | FuseMix 2023 |

Reference `R0_ref` = system E's recipe (copied from NB36). Stage A = usual folds, seeds 1-3; the top 2 are re-checked on NEW folds with NEW seeds (stage B); only stage B counts. The age-gender ViT features for the MAV "extra" rows are extracted once at the start (about 10 minutes; needs the MAV zips on Drive as in NB35).

**How to run:** Runtime > Change runtime type > T4 GPU, then Runtime > Run all. About 3.5 h. Resumable. Paste the **SUMMARY**.


In [ ]:
# Cell 1 - Drive, GPU (hard check)
from google.colab import drive
drive.mount('/content/drive')
import torch, shutil
assert torch.cuda.is_available(), 'NO GPU in this runtime. Runtime > Change runtime type > T4 GPU > Save, then run Cell 1 again.'
dev = 'cuda'
print('[NB37 v1 5-Nov] Device:', torch.cuda.get_device_name(0), '| disk free %.0f GB' % (shutil.disk_usage('/content')[2] / 1e9))

In [ ]:
# Cell 2 - features, labels, CV folds (identical to NB24), harness with the new recipe options
import os, re, json, time, zipfile, copy
import numpy as np, pandas as pd
import torch.nn as nn, torch.nn.functional as Fn
from sklearn.metrics import roc_curve
BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027'); FEATDIR = os.path.join(PROJECT, 'features'); RESDIR = os.path.join(PROJECT, 'results')
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)
LISTS = '/content/flag_lists'
if not os.path.exists(LISTS + '/done'):
    with zipfile.ZipFile(os.path.join(ZIP_DIR, 'train_set.zip')) as zf:
        zf.extractall(LISTS, members=[m for m in zf.namelist() if m.endswith(('.txt', '.csv'))])
    open(LISTS + '/done', 'w').close()
TXT = [os.path.join(r, f) for r, _, fs in os.walk(LISTS) for f in fs if f.endswith(('.txt', '.csv'))]
def find_one(*parts):
    hits = [p for p in TXT if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def read_feat(path, dim):
    df = pd.read_csv(path, header=None)
    return df.iloc[:, :dim].values.astype(np.float32), (df.iloc[:, dim].values if df.shape[1] > dim else None)
Xv, y = read_feat(find_one('train_English_voices.csv'), 192); y = y.astype(int)
train_lines = [l.split() for l in open(find_one('train_English.txt')) if l.strip()]
cls2spk = {int(l[5]): l[4] for l in train_lines}
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
ids = np.unique(y); id_gender = {c: spk2g[cls2spk[c]] for c in ids}
MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz'))); EXTRA = dict(np.load(os.path.join(FEATDIR, 'mavceleb_extra.npz')))
ML = np.load(os.path.join(FEATDIR, 'mav_lang.npz')); EXT_SPK, EXT_LANG = ML['spk'], ML['lang']
assert list(EXT_SPK) == list(MAV['spk']) + list(EXTRA['spk']), 'mav_lang.npz does not match the feature files - tell Claude'
FEAT = {'ecapa': {'train': Xv, 'mav': MAV['ecapa'], 'extra': EXTRA['ecapa']}}
for name in ['clip', 'wavlm_sv', 'r293', 'redim', 'audeer', 'whisper']:
    d = np.load(os.path.join(FEATDIR, name + '.npz')); FEAT[name] = {s: d[s] for s in d.files if not s.endswith('_ag')}
FEAT['clip']['mav'], FEAT['clip']['extra'] = MAV['clip'], EXTRA['clip']; FEAT['wavlm_sv']['mav'], FEAT['wavlm_sv']['extra'] = MAV['wavlm_sv'], EXTRA['wavlm_sv']
FEAT['r293']['extra'] = EXTRA['r293']
for name in FEAT:
    for s in ['train', 'mav', 'extra']:
        assert s in FEAT[name] and len(FEAT[name][s]) == {'train': len(y), 'mav': len(MAV['spk']), 'extra': len(EXTRA['spk'])}[s], '%s/%s missing or wrong length - tell Claude' % (name, s)
N_CH = len(y); ext_spk = list(EXT_SPK)
ext_lab = {s: 1000 + i for i, s in enumerate(sorted(set(ext_spk)))}
Y_ALL = np.r_[y, np.array([ext_lab[s] for s in ext_spk])]; D_ALL = np.r_[np.zeros(N_CH, int), np.ones(len(ext_spk), int)]; EXT = N_CH + np.arange(len(ext_spk))
def l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
def build(names, split):
    return np.concatenate([l2(FEAT[n][split]) for n in names], 1).astype(np.float32)
def build_train(names):
    return np.concatenate([build(names, 'train'), build(names, 'mav'), build(names, 'extra')])
NAP_K = 8
def nap_basis(V_ext, k):
    diffs = []
    for s in np.unique(EXT_SPK):
        m = EXT_SPK == s; langs = np.unique(EXT_LANG[m])
        if len(langs) < 2:
            continue
        mu = [V_ext[m & (EXT_LANG == l)].mean(0) for l in langs]
        for a in range(len(mu)):
            for b in range(a + 1, len(mu)):
                diffs.append(mu[a] - mu[b])
    Dm = np.stack(diffs); U = np.linalg.svd(Dm - Dm.mean(0), full_matrices=False)[2][:k]
    return U.astype(np.float32), len(diffs)
def nap_apply(X, U):
    return (X - (X @ U.T) @ U).astype(np.float32)
def eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr)); return (fpr[i] + fnr[i]) / 2 * 100
def make_pairs(idx, n=2000, same_gender_neg=False, seed=0):
    rng = np.random.default_rng(seed); spk_ = np.unique(y[idx]); rows_ = {s: idx[y[idx] == s] for s in spk_}
    V, F, L = [], [], []
    for _ in range(n):
        s = rng.choice(spk_); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[s])); L.append(1)
        s = rng.choice(spk_)
        cand = [t for t in spk_ if t != s and (not same_gender_neg or id_gender[t] == id_gender[s])] or [t for t in spk_ if t != s]
        t = rng.choice(cand); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[t])); L.append(0)
    return np.array(V), np.array(F), np.array(L)
K = 5; rng = np.random.default_rng(42); folds = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rng.shuffle(members)
    for i, c in enumerate(members):
        folds[i % K].append(c)
FOLD_DATA = []
for k in range(K):
    val_ids = set(folds[k]); va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_DATA.append((tr_idx, [make_pairs(va_idx, seed=k), make_pairs(va_idx, same_gender_neg=True, seed=100 + k)]))
_PROTO = {}
def max_sep_prototypes(n):
    if n in _PROTO:
        return _PROTO[n]
    P = np.array([[1.0, -1.0]])
    for k in range(2, n):
        Q = np.zeros((k, k + 1)); Q[0, 0] = 1.0; Q[0, 1:] = -1.0 / k; Q[1:, 1:] = np.sqrt(1 - 1.0 / k ** 2) * P; P = Q
    _PROTO[n] = torch.tensor(P.T, dtype=torch.float32); return _PROTO[n]
class Branch(nn.Module):
    def __init__(self, d_in, d=128):
        super().__init__(); self.net = nn.Sequential(nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(0.5))
    def forward(self, x):
        return Fn.normalize(self.net(x), dim=1)
class FOP(nn.Module):
    def __init__(self, n_class, d_face, d_voice, d=128):
        super().__init__()
        self.face, self.voice = Branch(d_face, d), Branch(d_voice, d)
        self.att = nn.Sequential(nn.Linear(2 * d, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, d))
        self.cls = nn.Linear(d, n_class - 1); self.register_buffer('P', max_sep_prototypes(n_class))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.constant_(m.bias, 0.01)
    def embed(self, xf, xv):
        f, v = self.face(xf), self.voice(xv); return torch.tanh(f), torch.tanh(v), f, v
    def forward(self, xf, xv):
        ft, vt, f, v = self.embed(xf, xv); a = torch.sigmoid(self.att(torch.cat([f, v], 1))); fused = ft * a + (1 - a) * vt
        return fused, self.cls(fused) @ self.P.t(), ft, vt
def opl_loss(feat, lab):
    feat = Fn.normalize(feat, dim=1); dot = feat @ feat.t(); same = (lab[:, None] == lab[None, :]).float()
    pos = same - torch.eye(len(lab), device=feat.device); neg = 1 - same
    return (1 - (pos * dot).sum() / (pos.sum() + 1e-6)) + 0.7 * (neg * dot.abs()).sum() / (neg.sum() + 1e-6)
def nce_loss(ft, vt, lab, tau):
    """symmetric identity-aware InfoNCE between the two scored embeddings: every same-identity row counts as a positive"""
    S = Fn.normalize(ft, dim=1) @ Fn.normalize(vt, dim=1).t() / tau; same = lab[:, None] == lab[None, :]
    def one(S_):
        logp = S_ - torch.logsumexp(S_, 1, keepdim=True); return -torch.logsumexp(logp.masked_fill(~same, -1e4), 1).mean()
    return 0.5 * (one(S) + one(S.t()))
@torch.no_grad()
def same_score(model, xf, xv):
    model.eval(); mf, sf_, mv, sv = model.norm; ft, vt, _, _ = model.embed((xf - mf) / sf_, (xv - mv) / sv)
    return -(ft - vt).norm(dim=1).cpu().numpy()
def dnorm(F_raw, V_raw, tr_idx):
    F, V = F_raw.copy(), V_raw.copy(); ch = None
    for d in np.unique(D_ALL[tr_idx]):
        trd = tr_idx[D_ALL[tr_idx] == d]; alld = np.where(D_ALL == d)[0]
        st = (F_raw[trd].mean(0), F_raw[trd].std(0) + 1e-6, V_raw[trd].mean(0), V_raw[trd].std(0) + 1e-6)
        F[alld] = (F_raw[alld] - st[0]) / st[1]; V[alld] = (V_raw[alld] - st[2]) / st[3]
        if d == 0:
            ch = st
    return F, V, ch
SEEDS = [1, 2, 3]; P_ID, K_CLIP, ALPHA, LR, MAX_EP, PATIENCE, STEPS = 16, 8, 2.0, 1e-4, 100, 15, 60
def train_head(F_raw, V_raw, all_tr, vsets=None, seed=1, cfg=None, max_epochs=MAX_EP):
    """cfg keys: align in {None,'mse','nce'}, w (alignment weight), tau, repair (bool), p_in (input dropout)"""
    cfg = cfg or {}; align, w_al, tau, repair, p_in = cfg.get('align'), cfg.get('w', 1.0), cfg.get('tau', 0.07), cfg.get('repair', False), cfg.get('p_in', 0.0)
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    F, V, ch = dnorm(F_raw, V_raw, all_tr); XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev); pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    tr = torch.tensor(all_tr, device=dev); model = FOP(len(cl), XF.shape[1], XV.shape[1]).to(dev)
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6); mf, sf_, mv, sv = model.norm
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01); best, best_ep, best_state, wait = 1e9, max_epochs, None, 0
    for ep in range(1, max_epochs + 1):
        model.train()
        for _ in range(STEPS):
            chosen = rng.choice(len(cl), P_ID, replace=False)
            j_np = np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen])
            jv_np = np.concatenate([g[rng.permutation(K_CLIP)] for g in j_np.reshape(P_ID, K_CLIP)]) if repair else j_np   # voice from another clip of the same identity
            j, jv = torch.tensor(j_np, device=dev), torch.tensor(jv_np, device=dev)
            xf = (XF[tr[j]] - mf) / sf_; xv = (XV[tr[jv]] - mv) / sv
            if p_in > 0:
                xf = Fn.dropout(xf, p_in, True); xv = Fn.dropout(xv, p_in, True)
            fused, logits, ft, vt = model(xf, xv); lab = ytr[j]
            loss = Fn.cross_entropy(logits, lab) + ALPHA * opl_loss(fused, lab)
            if align == 'mse':
                loss = loss + w_al * ((ft - vt) ** 2).mean()
            elif align == 'nce':
                loss = loss + w_al * nce_loss(ft, vt, lab, tau)
            opt.zero_grad(); loss.backward(); opt.step()
        if vsets is None:
            continue
        score = np.mean([eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_ep, XF, XV, ch
R4 = ['ecapa', 'wavlm_sv', 'r293', 'redim', 'audeer']; VOICE = R4 + ['whisper']
F_RAW = build_train(['clip']); V_full = build_train(VOICE); U_NAP, n_d = nap_basis(V_full[N_CH:], NAP_K); V_RAW = nap_apply(V_full, U_NAP)
note('harness ready: %d challenge rows, %d MAV rows, face %d-d, voice %d-d after NAP-%d (%d language-difference vectors)' % (N_CH, len(EXT), F_RAW.shape[1], V_RAW.shape[1], NAP_K, n_d))

In [ ]:
# NB37 - research round 7 on top of system E. Needs the two setup cells above.
# Five things the FAME 2024/2026 winners did that we have not tried, each judged with the usual CV (stage A) and re-checked on NEW folds + NEW seeds (stage B):
#   F_agv   : concatenate the frozen age-gender ViT embedding (abhilash88/age-gender-prediction CLS, 768-d = the FAME 2026 winner's face attribute model) to CLIP on the face side
#   L_sf2   : SphereFace2-style binary loss per prototype instead of softmax CE on the fixed max-separation prototypes (Han et al., ICASSP 2023)
#   L_hinge : margin-relaxed cosine hinge between the scored face and voice embeddings (zero gradient inside 30 degrees; Cross-Modal Distillation 2025)
#   H_pair  : a small pair classifier on [f, v, f*v, |f-v|] trained with BCE on in-batch pairs, scored by its logit (the FAME 2024 winner's scorer)
#   X_mix   : FuseMix-style within-identity latent mixup with a shared lambda on face and voice (same identity -> label kept)
# Reference R0_ref = system E's recipe, copied from NB36 (M0_ref rows, stage A and B). Training-side only; per-trial scoring; no test statistics.
SUMMARY = []
# ---------------- age-gender ViT features: challenge + MAV v1/v3 exist (NB12 agv.npz); MAV 'extra' rows are extracted here once ----------------
AGV_PATH, AGV_EXTRA = os.path.join(FEATDIR, 'agv.npz'), os.path.join(FEATDIR, 'agv_extra.npy')
agv = np.load(AGV_PATH); assert len(agv['train']) == N_CH and len(agv['mav']) == len(MAV['spk']), 'agv.npz rows do not match the features - tell Claude'
if not os.path.exists(AGV_EXTRA):
    from PIL import Image
    from tqdm.auto import tqdm
    from transformers import AutoModel, AutoImageProcessor
    MAV_ZIPS, MAV_LISTS = os.path.join(BASE, 'dataset'), os.path.join(BASE, 'face_voice_association_splits')
    EXTRA_SPK = EXTRA['spk']; MAX_PER_ID = 120
    VPAT = re.compile(r'voices/(id\d+)/([^/]+)/([^/]+)/([^/]+\.wav)$'); FPAT = re.compile(r'faces/(id\d+)/([^/]+)/([^/]+)/([^/]+\.jpg)$')
    v1_used = set()
    for lf in ['English_train.txt', 'Urdu_train.txt']:
        for l in open(os.path.join(MAV_LISTS, 'v1', lf)):
            t = l.split()
            if t:
                v1_used.add(t[0].split('/')[0])
    rows = []
    for ver, zname, skip_ids in [('v2', 'mavceleb_v2_complete.zip', set()), ('v1', 'mavceleb_v1_complete.zip', v1_used)]:
        D = '/content/mavxf/' + ver; flag = os.path.join(D, 'faces.done'); need = not os.path.exists(flag)
        zf = zipfile.ZipFile(os.path.join(MAV_ZIPS, zname)); voices, faces = {}, {}
        for n in zf.namelist():
            mv = VPAT.search(n)
            if mv:
                voices.setdefault(mv.group(1), []).append((mv.group(2), mv.group(3), n)); continue
            mf = FPAT.search(n)
            if mf:
                faces.setdefault((mf.group(1), mf.group(2), mf.group(3)), []).append(n)
        ids_ = sorted(i for i in voices if i not in skip_ids); rng0 = np.random.default_rng(0)
        for pid in ids_:
            vs = voices[pid]; vs = [vs[j] for j in rng0.permutation(len(vs))[:MAX_PER_ID]]
            for lang_, vid, vn in vs:
                fr = faces.get((pid, lang_, vid)) or [f for (p, l, v), fl in faces.items() if p == pid for f in fl]
                if not fr:
                    continue
                fn = fr[rng0.integers(len(fr))]
                if need:
                    zf.extract(fn, D)
                rows.append((ver + '_' + pid, os.path.join(D, fn)))
        zf.close()
        if need:
            open(flag, 'w').close()
    spk_all = np.array([r[0] for r in rows]); keep = np.ones(len(rows), bool)
    if len(rows) != len(EXTRA_SPK):
        j = 0; keep[:] = False
        for i in range(len(rows)):
            if j < len(EXTRA_SPK) and spk_all[i] == EXTRA_SPK[j]:
                keep[i] = True; j += 1
        assert j == len(EXTRA_SPK), 'could not align MAV extra rows - tell Claude'
    extra_files = [r[1] for r, k in zip(rows, keep) if k]
    assert [r[0] for r, k in zip(rows, keep) if k] == list(EXTRA_SPK), 'MAV extra row order differs - tell Claude'
    name_ = 'abhilash88/age-gender-prediction'
    model_ = AutoModel.from_pretrained(name_, trust_remote_code=True).to(dev).eval()
    try:
        proc_ = AutoImageProcessor.from_pretrained(name_, trust_remote_code=True)
    except Exception:
        proc_ = AutoImageProcessor.from_pretrained('google/vit-base-patch16-224')
    proc_.do_center_crop = False; proc_.size = {'height': 224, 'width': 224}
    backbone_ = getattr(model_, 'vit', None) or getattr(model_, 'backbone', None) or model_
    out_ = []
    for i in tqdm(range(0, len(extra_files), 64), desc='age-gender ViT (extra)'):
        ims = [Image.open(p).convert('RGB') for p in extra_files[i:i + 64]]
        x = proc_(images=ims, return_tensors='pt')['pixel_values'].to(dev)
        with torch.no_grad():
            o = backbone_(pixel_values=x)
        h = o.last_hidden_state[:, 0] if getattr(o, 'last_hidden_state', None) is not None else torch.as_tensor(o)
        out_.append(h.float().cpu().numpy())
    np.save(AGV_EXTRA, np.concatenate(out_).astype(np.float32)); del model_; torch.cuda.empty_cache()
    note('age-gender ViT extracted for %d MAV extra faces' % len(extra_files))
AGV_RAW = np.concatenate([l2(agv['train']), l2(agv['mav']), l2(np.load(AGV_EXTRA))]).astype(np.float32); assert len(AGV_RAW) == len(F_RAW)
F_AGV = np.concatenate([F_RAW, AGV_RAW], 1)
# sanity: can the age-gender embedding separate identities at all, and how much does it correlate with voice age?
note('face inputs: clip %d-d | clip+agv %d-d' % (F_RAW.shape[1], F_AGV.shape[1]))
# ---------------- losses ----------------
def lout_loss(ft, vt, lab, tau):
    S = Fn.normalize(ft, dim=1) @ Fn.normalize(vt, dim=1).t() / tau; P = (lab[:, None] == lab[None, :]).float()
    def one(M):
        return -(P * Fn.log_softmax(M, 1)).sum(1).div(P.sum(1)).mean()
    return 0.5 * (one(S) + one(S.t()))
def sf2_loss(z, P, lab, bias, s=16.0, m=0.1, t=3.0, lam=0.7):
    """SphereFace2: one binary decision per prototype; positive prototype pulled past +m, every other prototype pushed below -m"""
    cos = Fn.normalize(z, dim=1) @ P.t(); g = 2 * ((cos + 1) / 2) ** t - 1
    onehot = Fn.one_hot(lab, g.shape[1]).float()
    pos = Fn.softplus(-s * (g - m) - bias) * onehot; neg = Fn.softplus(s * (g + m) + bias) * (1 - onehot)
    return (lam * pos.sum(1) + (1 - lam) * neg.sum(1) / (g.shape[1] - 1)).mean()
def hinge_loss(ft, vt, m=1 - np.cos(np.deg2rad(30))):
    return Fn.relu(1 - (Fn.normalize(ft, dim=1) * Fn.normalize(vt, dim=1)).sum(1) - m).mean()
class PairHead(nn.Module):
    def __init__(self, d=128):
        super().__init__(); self.net = nn.Sequential(nn.Linear(4 * d, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, 1))
    def forward(self, f, v):
        return self.net(torch.cat([f, v, f * v, (f - v).abs()], 1)).squeeze(1)
def pair_bce(ph, ft, vt, lab):
    n = len(lab); same = lab[:, None] == lab[None, :]
    pi, pj = torch.where(same); ni, nj = torch.where(~same)
    sel = torch.randperm(len(ni), device=lab.device)[:len(pi)]; ni, nj = ni[sel], nj[sel]
    logits = torch.cat([ph(ft[pi], vt[pj]), ph(ft[ni], vt[nj])]); y = torch.cat([torch.ones(len(pi)), torch.zeros(len(ni))]).to(lab.device)
    return Fn.binary_cross_entropy_with_logits(logits, y)
@torch.no_grad()
def pair_score(model, xf, xv):
    model.eval(); mf, sf_, mv, sv = model.norm; ft, vt, _, _ = model.embed((xf - mf) / sf_, (xv - mv) / sv)
    return model.pair(ft, vt).cpu().numpy()
def train_head7(F_raw, V_raw, all_tr, vsets, seed, cfg):
    w_al, tau = 0.5, 0.07
    use_sf2, w_h, use_pair, mix_p = cfg.get('sf2', False), cfg.get('hinge', 0.0), cfg.get('pair', False), cfg.get('mix', 0.0)
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    F, V, ch = dnorm(F_raw, V_raw, all_tr); XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev); pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    tr = torch.tensor(all_tr, device=dev); model = FOP(len(cl), XF.shape[1], XV.shape[1]).to(dev)
    if use_pair:
        model.pair = PairHead(128).to(dev)
    model.sf2_bias = nn.Parameter(torch.zeros(1, device=dev))
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6); mf, sf_, mv, sv = model.norm
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01); best, best_ep, best_state, wait = 1e9, MAX_EP, None, 0
    for ep in range(1, MAX_EP + 1):
        model.train()
        for _ in range(STEPS):
            chosen = rng.choice(len(cl), P_ID, replace=False)
            j_np = np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen]); j = torch.tensor(j_np, device=dev)
            xf = (XF[tr[j]] - mf) / sf_; xv = (XV[tr[j]] - mv) / sv; lab = ytr[j]
            if mix_p > 0:
                # partner = another row of the same identity (the 8 rows of each identity are contiguous); shared lambda per row
                part = torch.tensor(np.concatenate([g[rng.permutation(K_CLIP)] for g in np.arange(len(j_np)).reshape(P_ID, K_CLIP)]), device=dev)
                lam = torch.tensor(rng.beta(2.0, 2.0, size=len(j_np)), device=dev, dtype=torch.float32)
                lam = torch.where(torch.tensor(rng.random(len(j_np)) < mix_p, device=dev), lam, torch.ones_like(lam))[:, None]
                xf = lam * xf + (1 - lam) * xf[part]; xv = lam * xv + (1 - lam) * xv[part]
            fused, logits, ft, vt = model(xf, xv)
            cls_term = sf2_loss(model.cls(fused), model.P, lab, model.sf2_bias) if use_sf2 else Fn.cross_entropy(logits, lab)
            loss = cls_term + ALPHA * opl_loss(fused, lab) + w_al * lout_loss(ft, vt, lab, tau)
            if w_h > 0:
                loss = loss + w_h * hinge_loss(ft, vt)
            if use_pair:
                loss = loss + pair_bce(model.pair, ft, vt, lab)
            opt.zero_grad(); loss.backward(); opt.step()
        scorer = pair_score if use_pair else same_score
        score = np.mean([eer(L, scorer(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    model.load_state_dict(best_state); return model, best_ep, XF, XV
CFGS = {'F_agv': dict(face='agv'), 'L_sf2': dict(sf2=True), 'L_hinge': dict(hinge=1.0), 'H_pair': dict(pair=True), 'X_mix': dict(mix=0.5)}
rngB = np.random.default_rng(7); foldsB = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rngB.shuffle(members)
    for i, c in enumerate(members):
        foldsB[i % K].append(c)
FOLD_B = []
for k in range(K):
    val_ids = set(foldsB[k]); va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_B.append((tr_idx, [make_pairs(va_idx, seed=200 + k), make_pairs(va_idx, same_gender_neg=True, seed=300 + k)]))
CSV37 = os.path.join(RESDIR, 'nb37_cv.csv')
if not os.path.exists(CSV37):
    r = pd.read_csv(os.path.join(RESDIR, 'nb36_cv.csv')); r = r[r.combo == 'M0_ref'][['stage', 'combo', 'seed', 'fold', 'best_ep', 'std', 'gender']].copy()
    r['combo'] = 'R0_ref'; r['std_l2'] = r['std']; r['gender_l2'] = r['gender']; r.to_csv(CSV37, index=False)
    print('R0_ref copied from NB36 M0_ref (%d runs: stage A %d, stage B %d)' % (len(r), (r.stage == 'A').sum(), (r.stage == 'B').sum()))
done37 = pd.read_csv(CSV37); t0 = time.time()
def run37(stage, names, seeds, fold_data):
    global done37
    for name in names:
        for seed in seeds:
            for k in range(K):
                if ((done37.stage == stage) & (done37.combo == name) & (done37.seed == seed) & (done37.fold == k)).any():
                    continue
                tr_idx, vsets = fold_data[k]; cfg = CFGS[name]; Fin = F_AGV if cfg.get('face') == 'agv' else F_RAW
                try:
                    model, best_ep, XF, XV = train_head7(Fin, V_RAW, np.concatenate([tr_idx, EXT]), vsets, seed, cfg)
                    l2_std, l2_gen = [eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets]
                    if cfg.get('pair'):
                        e_std, e_gen = [eer(L, pair_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets]
                    else:
                        e_std, e_gen = l2_std, l2_gen
                except Exception as e:
                    print(name, 'FAILED', repr(e)[:300]); best_ep, e_std, e_gen, l2_std, l2_gen = -1, 50.0, 50.0, 50.0, 50.0
                row = pd.DataFrame([dict(stage=stage, combo=name, seed=seed, fold=k, best_ep=best_ep, std=e_std, gender=e_gen, std_l2=l2_std, gender_l2=l2_gen)])
                row.to_csv(CSV37, mode='a', header=False, index=False); done37 = pd.concat([done37, row])
            print('%s %-8s seed %d done | %.0f min' % (stage, name, seed, (time.time() - t0) / 60))
def tab37(stage):
    R = pd.read_csv(CSV37); R = R[R.stage == stage].copy(); R['overall'] = (R['std'] + R['gender']) / 2; R['overall_l2'] = (R['std_l2'] + R['gender_l2']) / 2
    return R.groupby(['combo', 'seed'])[['std', 'gender', 'overall', 'overall_l2', 'best_ep']].mean().groupby('combo').mean().round(2).sort_values('overall')
run37('A', list(CFGS), [1, 2, 3], FOLD_DATA)
TA = tab37('A'); refA = TA.loc['R0_ref']; TOP = [c for c in TA.index if c != 'R0_ref'][:2]
run37('B', TOP, [4, 5, 6], FOLD_B)
TB = tab37('B'); refB = TB.loc['R0_ref']
note('NB37 STAGE A (usual folds, seeds 1-3; R0_ref = system E recipe from NB36; "overall_l2" = -L2 score of the same head, differs from "overall" only for H_pair):'); note(TA.to_string())
note('NB37 STAGE B (NEW folds + NEW seeds 4-6):'); note(TB.to_string())
note('VERDICTS (stage B only: overall <= -0.4, or gender <= -0.5 with std <= +0.2 vs R0_ref):')
for c in TOP:
    dB, dg, ds = TB.loc[c, 'overall'] - refB['overall'], TB.loc[c, 'gender'] - refB['gender'], TB.loc[c, 'std'] - refB['std']
    note('%s %s: A %+.2f | B %+.2f (std %+.2f, gender %+.2f) -> %s' % (c, json.dumps(CFGS[c]), TA.loc[c, 'overall'] - refA['overall'], dB, ds, dg, 'ADOPT' if (dB <= -0.4 or (dg <= -0.5 and ds <= 0.2)) else 'reject'))
print('=' * 70); print('SUMMARY  (copy everything between the lines and paste to Claude)'); print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)
